# Bronze -- `bronze_commerce_products`

Landing only. No deduplication, no filtering, no business logic.

**Source:** `commerce.products`  
**File:** `products.csv`  
**Watermark:** `updated_at`  
**Load pattern:** `incremental`

> Every upstream defect survives this layer intact. If bronze silently fixed anything, a silver bug would be indistinguishable from an upstream change and replay would not reproduce the original state.

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
target_item = "lh_bronze"
source_item = "lh_bronze"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="bronze_commerce_products",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="bronze", table_name="bronze_commerce_products")
print(f"load_id={load_id}  environment={environment}  table=bronze_commerce_products")


In [ ]:
# ---- Declared schema ---------------------------------------------
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, LongType, BooleanType, DateType)

schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", StringType(), True),
    StructField("stock", IntegerType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])


In [ ]:
# ---- Read the landed file ----------------------------------------
landing_path = "Files/bronze/commerce/products"

df = (spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .option("encoding", "utf-8")
    .option("quote", '"')
    .schema(schema)
    .csv(landing_path))

rows_in = df.count()
dq.record_input(rows_in)
print(f"read {rows_in:,} rows from {landing_path}")


In [ ]:
# ---- Landing checks ----------------------------------------------
# These test whether the file ARRIVED correctly -- not whether its
# contents are any good, which is silver's job.
#
# The header is re-read WITHOUT the declared schema. Reading it from
# `df.columns` would return the schema's own names, so the check would
# compare the schema to itself and could never fail -- while the
# schema, applied positionally, silently loaded values into the wrong
# columns. That failure is invisible whenever the mismatched columns
# share a type.
expected = [f.name for f in schema.fields]
actual = (spark.read
    .option("header", "true")
    .csv(landing_path)
    .columns)

if actual != expected:
    raise AssertionError(
        f"header_matches_registry failed.\n"
        f"  registry: {expected}\n"
        f"  file:     {actual}\n"
        f"Order matters -- the schema is applied positionally."
    )
if rows_in == 0:
    raise AssertionError("row_count_not_zero failed: a zero-row file "
                         "almost always means a broken export")
print("landing checks passed")


In [ ]:
# ---- Incremental watermark ---------------------------------------
# Bronze appends and never overwrites, so without this filter every
# re-run re-appends the entire source and silently multiplies the
# table. Deduplication downstream would hide it, which is worse than
# failing -- the counts stay plausible while bronze stops being a
# faithful record of what arrived.
from pyspark.sql.utils import AnalysisException

try:
    high_water = (spark.read.table("bronze_commerce_products")
        .agg(F.max("updated_at")).collect()[0][0])
except AnalysisException:
    high_water = None          # first load; take everything

if high_water:
    before = rows_in
    df = df.filter(F.col("updated_at") > F.lit(high_water))
    rows_in = df.count()
    print(f"watermark updated_at > {high_water}: "
          f"{before:,} -> {rows_in:,} new rows")
else:
    print("no high-water mark; loading all {rows_in:,} rows".format(rows_in=rows_in))

dq.record_input(rows_in)

if rows_in == 0:
    print("nothing new to land")
    dq.record_output(0)
    dq.flush()
    mssparkutils.notebook.exit("no new rows")


In [ ]:
# ---- Audit columns and write -------------------------------------
# Injected from 00-platform.yaml `audit_columns.bronze`, so the
# provenance contract is identical across every entity.
out = (df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit("commerce"))
    .withColumn("_load_id", F.lit(load_id))
    .withColumn("_source_file", F.input_file_name())
)

out.write.mode("append") \
    .format("delta").saveAsTable("bronze_commerce_products")

dq.record_output(rows_in)
print(f"landed {rows_in:,} rows into bronze_commerce_products")
dq.flush()
